# This code is working fine slicing, reverse slicing, geo-location csv testing

In [24]:
import os
import numpy as np
import rasterio
from rasterio.windows import Window
from rasterio.transform import from_origin
from PIL import Image

def slice_geotiff(geotiff_path, output_dir, output_size, slice_flag=True, to_jpeg=True):
    """
    Slice a GeoTIFF color image into smaller square images or convert it to a single JPEG.

    Args:
        geotiff_path (str): Path to the input GeoTIFF file.
        output_dir (str): Directory to save the output slices or image.
        output_size (int): Size (in pixels) of each square output slice.
        slice_flag (bool): If True, slices the image into square tiles. If False, no slicing is performed.
        to_jpeg (bool): If True, outputs JPEG images; otherwise, retains TIFF format with geospatial metadata.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    with rasterio.open(geotiff_path) as src:
        img_array = src.read()  # Read all bands (multi-band support)
        height, width = src.height, src.width
        transform = src.transform  # Get geotransform
        crs = src.crs  # Get coordinate reference system
        dtype = src.dtypes[0]  # Data type
        count = src.count  # Number of bands
        
        # Determine the number of tiles in each dimension
        num_tiles_x = int(np.ceil(width / output_size))
        num_tiles_y = int(np.ceil(height / output_size))
        
        if not slice_flag:
            # Save the entire image as a single output
            img = np.moveaxis(img_array, 0, -1)  # Convert to HxWxBands
            img = Image.fromarray(img)
            if img.mode == 'RGBA' and to_jpeg:
                img = img.convert('RGB')  # Convert to RGB for JPEG format
            output_format = "JPEG" if to_jpeg else "TIFF"
            img.save(os.path.join(output_dir, f"output.{output_format.lower()}"), output_format)
            return
        
        # Slice into smaller square images
        for y in range(num_tiles_y):
            for x in range(num_tiles_x):
                # Calculate slice boundaries
                x_start = x * output_size
                y_start = y * output_size
                x_end = min(x_start + output_size, width)
                y_end = min(y_start + output_size, height)
                
                # Read windowed data
                window = Window(x_start, y_start, x_end - x_start, y_end - y_start)
                slice_img = src.read(window=window)
                
                # Pad if necessary to maintain square size
                pad_x = output_size - (x_end - x_start)
                pad_y = output_size - (y_end - y_start)
                if pad_x > 0 or pad_y > 0:
                    slice_img = np.pad(slice_img, ((0, 0), (0, pad_y), (0, pad_x)), mode='constant', constant_values=0)
                
                # Calculate new transform for the sliced image
                new_transform = from_origin(
                    transform.c + x_start * transform.a,  # Upper-left x-coordinate
                    transform.f + y_start * transform.e,  # Upper-left y-coordinate
                    transform.a,  # Pixel width
                    -transform.e   # Pixel height
                )
                
                # Save the slice with metadata
                filename = f"slice_{y}_{x}.jpg" if to_jpeg else f"slice_{y}_{x}.tif"
                output_path = os.path.join(output_dir, filename)
                
                if to_jpeg:
                    img = np.moveaxis(slice_img, 0, -1)
                    img = Image.fromarray(img)
                    if img.mode == 'RGBA':
                        img = img.convert('RGB')
                    img.save(output_path, "JPEG")
                else:
                    with rasterio.open(
                        output_path, "w",
                        driver="GTiff",
                        height=output_size,
                        width=output_size,
                        count=count,
                        dtype=dtype,
                        crs=crs,
                        transform=new_transform
                    ) as dst:
                        dst.write(slice_img)


In [32]:
import os
import numpy as np
import rasterio
from rasterio.transform import from_origin

def reverse_slice_geotiff(input_dir, output_geotiff):
    """
    Reassemble square GeoTIFF image slices into a single image, discarding black pixels.

    Args:
        input_dir (str): Directory containing the sliced GeoTIFF images.
        output_geotiff (str): Path to save the reassembled GeoTIFF file.
    """
    slice_files = sorted([f for f in os.listdir(input_dir) if f.endswith(".tif")])
    if not slice_files:
        raise ValueError("No TIFF slices found in the input directory.")
    
    def extract_indices(filename):
        """ Extract x and y indices from filename like 'slice_y_x.tif' """
        try:
            parts = filename.split("_")
            y, x = int(parts[1]), int(parts[2].split(".")[0])
            return y, x
        except (IndexError, ValueError):
            raise ValueError(f"Invalid filename format: {filename}")

    # Extract tile grid size
    tile_indices = [extract_indices(f) for f in slice_files]
    num_tiles_x = max(idx[1] for idx in tile_indices) + 1
    num_tiles_y = max(idx[0] for idx in tile_indices) + 1

    # Read first tile for metadata
    with rasterio.open(os.path.join(input_dir, slice_files[0])) as src:
        slice_height, slice_width = src.height, src.width
        count, dtype, crs, transform = src.count, src.dtypes[0], src.crs, src.transform
        
    # Create empty array for reassembled image
    full_height = num_tiles_y * slice_height
    full_width = num_tiles_x * slice_width
    full_image = np.zeros((count, full_height, full_width), dtype=dtype)

    # Fill in tiles
    for f in slice_files:
        y, x = extract_indices(f)
        with rasterio.open(os.path.join(input_dir, f)) as tile:
            tile_data = tile.read()
            mask = np.any(tile_data != 0, axis=0)  # Mask out black pixels
            for band in range(count):
                full_image[band, y * slice_height:(y + 1) * slice_height, x * slice_width:(x + 1) * slice_width][mask] = tile_data[band][mask]

    # Correct geotransform
    new_transform = from_origin(transform.c, transform.f, transform.a, -transform.e)

    # Save the reassembled image
    with rasterio.open(
        output_geotiff, "w",
        driver="GTiff",
        height=full_height,
        width=full_width,
        count=count,
        dtype=dtype,
        crs=crs,
        transform=new_transform
    ) as dst:
        dst.write(full_image)


In [35]:
import os

# Define paths and parameters
geotiff_path = "m_3008501_ne_16_030_20211103.tif"  # Path to the input GeoTIFF file
output_dir = "output_slices_geo"           # Directory to save sliced images
output_dir_normal = "output_slices"           # Directory to save sliced images
reassembled_output = "reassembled_output_tiff_1.tif"  # Path for the reassembled image

output_size = 3000  # Size of each square slice in pixels
slice_flag = True  # Enable slicing

# Step 1: Slice the GeoTIFF image into tiles
print("Slicing the GeoTIFF image...")
slice_geotiff(
    geotiff_path=geotiff_path,
    output_dir=output_dir,
    output_size=output_size,
    slice_flag=slice_flag,  # Enable slicing
    to_jpeg=False            # Save slices as JPEG images
)

print("Slicing the normal image...")
slice_geotiff(
    geotiff_path=geotiff_path,
    output_dir=output_dir_normal,
    output_size=output_size,
    slice_flag=slice_flag,  # Enable slicing
    to_jpeg=True            # Save slices as JPEG images
)

print("Reassembling the image from slices...")
reverse_slice_geotiff(input_dir=output_dir, output_geotiff=reassembled_output)

Slicing the GeoTIFF image...
Slicing the normal image...
Reassembling the image from slices...


In [43]:
# import os
# import cv2
# import glob
# import torch
# import matplotlib.pyplot as plt
# from ultralytics import YOLO
# from PIL import Image

# # Load the YOLO model
# model = YOLO("C:/Users/mza0288/Documents/CODE/runs/segment/train29/weights/best.pt")

# # Directory containing images
# image_dir = "C:/Users/mza0288/Documents/CODE/output_slices"  # Change to your image directory
# image_paths = glob.glob(os.path.join(image_dir, "*.jpg")) + glob.glob(os.path.join(image_dir, "*.png")) + glob.glob(os.path.join(image_dir, "*.jpeg"))

# # Function to display images
# def display_images(original, predicted):
#     fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    
#     # Show Original Image
#     axes[0].imshow(original)
#     axes[0].set_title("Original Image")
#     axes[0].axis("off")
    
#     # Show Predicted Image
#     axes[1].imshow(predicted)
#     axes[1].set_title("Predicted Image")
#     axes[1].axis("off")
    
#     plt.show()

# # Process each image
# for image_path in image_paths:
#     # Load original image
#     img = Image.open(image_path).convert("RGB")
    
#     # Perform segmentation
#     results = model(image_path)

#     # Extract the predicted image
#     pred_img = results[0].plot()  # Get segmentation overlay

#     # Convert OpenCV BGR to RGB for proper display
#     pred_img = cv2.cvtColor(pred_img, cv2.COLOR_BGR2RGB)
    
#     # Display the original and predicted images
#     display_images(img, pred_img)


In [44]:
# import os
# import cv2
# import glob
# import numpy as np
# import torch
# import matplotlib.pyplot as plt
# from ultralytics import YOLO
# from PIL import Image, ImageDraw, ImageFont

# # Load the YOLO instance segmentation model
# model = YOLO("C:/Users/mza0288/Documents/CODE/runs/segment/train29/weights/best.pt")

# # Directory containing images
# image_dir = "C:/Users/mza0288/Documents/CODE/output_slices"  # Change to your directory
# image_paths = glob.glob(os.path.join(image_dir, "*.jpg")) + glob.glob(os.path.join(image_dir, "*.png")) + glob.glob(os.path.join(image_dir, "*.jpeg"))

# # Function to overlay masks, bounding boxes, and labels
# def draw_segmentation_results(image, results, alpha=0.5):
#     """
#     Draws segmentation masks, bounding boxes, and labels on the image.
    
#     Args:
#     - image: Original image (PIL Image format).
#     - results: YOLO model results.
#     - alpha: Transparency factor for the mask overlay (0=transparent, 1=opaque).
    
#     Returns:
#     - Annotated image (PIL Image format).
#     """
#     img_np = np.array(image)  # Convert PIL image to NumPy array
#     overlay = img_np.copy()

#     for result in results:
#         if result.masks is None or result.boxes is None:
#             print("Warning: No segmentation masks found for this image.")
#             return image  # Return the original image without changes

#         for mask, box, cls in zip(result.masks.xy, result.boxes.xyxy, result.boxes.cls):
#             # Convert class index to integer
#             cls = int(cls.item())

#             # Define a fixed color for all detected objects
#             color = (0, 255, 0)  # Green color for all masks

#             # Draw segmentation mask
#             mask_np = np.zeros_like(img_np, dtype=np.uint8)
#             cv2.fillPoly(mask_np, [np.array(mask, dtype=np.int32)], color)
            
#             # Blend mask with image
#             img_np = cv2.addWeighted(img_np, 1, mask_np, alpha, 0)

#             # Draw bounding box
#             x1, y1, x2, y2 = map(int, box.tolist())
#             cv2.rectangle(img_np, (x1, y1), (x2, y2), color, 2)

#             # Draw label beside the object
#             label = f"Class {cls}"
#             text_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)[0]
#             text_x = x2 + 5  # Offset label position to the right of the object
#             text_y = y1 + text_size[1]
            
#             cv2.putText(img_np, label, (text_x, text_y), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

#     return Image.fromarray(img_np)  # Convert NumPy array back to PIL Image

# # Function to display images
# def display_images(original, predicted):
#     fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    
#     # Show Original Image
#     axes[0].imshow(original)
#     axes[0].set_title("Original Image")
#     axes[0].axis("off")
    
#     # Show Predicted Image
#     axes[1].imshow(predicted)
#     axes[1].set_title("Predicted Image")
#     axes[1].axis("off")
    
#     plt.show()

# # Process each image
# for image_path in image_paths:
#     # Load original image
#     img = Image.open(image_path).convert("RGB")
    
#     # Perform segmentation
#     results = model(image_path)

#     # Draw results with more control
#     pred_img = draw_segmentation_results(img, results)

#     # Display the original and predicted images
#     display_images(img, pred_img)


In [70]:
# import os
# import cv2
# import glob
# import numpy as np
# import torch
# import matplotlib.pyplot as plt
# from ultralytics import YOLO
# import rasterio
# from rasterio.transform import from_origin
# from PIL import Image, ImageDraw, ImageFont

# # Load the YOLO instance segmentation model
# model = YOLO("C:/Users/mza0288/Documents/CODE/runs/segment/train29/weights/best.pt")

# # Directories
# image_dir = "C:/Users/mza0288/Documents/CODE/output_slices"  # RGB image directory
# geotiff_dir = "C:/Users/mza0288/Documents/CODE/output_slices_geo"  # GeoTIFF directory
# output_geotiff_dir = "C:/Users/mza0288/Documents/CODE/output_geotiffs_yolo"  # Output directory for modified GeoTIFFs

# os.makedirs(output_geotiff_dir, exist_ok=True)

# image_paths = glob.glob(os.path.join(image_dir, "*.jpg")) + glob.glob(os.path.join(image_dir, "*.png")) + glob.glob(os.path.join(image_dir, "*.jpeg"))

# def draw_segmentation_results(image, results, alpha=0.5):
#     """
#     Draws segmentation masks, bounding boxes, and labels on the image.
    
#     Args:
#     - image: Original image (PIL Image format).
#     - results: YOLO model results.
#     - alpha: Transparency factor for the mask overlay (0=transparent, 1=opaque).
    
#     Returns:
#     - Annotated image (PIL Image format).
#     """
#     img_np = np.array(image)  # Convert PIL image to NumPy array

#     for result in results:
#         if result.masks is None or result.boxes is None:
#             print("Warning: No segmentation masks found for this image.")
#             return image  # Return the original image without changes

#         for mask, box, cls in zip(result.masks.xy, result.boxes.xyxy, result.boxes.cls):
#             # Convert class index to integer
#             cls = int(cls.item())

#             # Define a fixed color for all detected objects
#             color = (0, 255, 0)  # Green color for all masks

#             # Draw segmentation mask
#             mask_np = np.zeros_like(img_np, dtype=np.uint8)
#             cv2.fillPoly(mask_np, [np.array(mask, dtype=np.int32)], color)
            
#             # Blend mask with image
#             img_np = cv2.addWeighted(img_np, 1, mask_np, alpha, 0)

#             # Draw bounding box
#             x1, y1, x2, y2 = map(int, box.tolist())
#             cv2.rectangle(img_np, (x1, y1), (x2, y2), color, 2)

#             # Draw label beside the object
#             label = f"Class {cls}"
#             text_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)[0]
#             text_x = x2 + 5  # Offset label position to the right of the object
#             text_y = y1 + text_size[1]
            
#             cv2.putText(img_np, label, (text_x, text_y), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

#     return Image.fromarray(img_np)  # Convert NumPy array back to PIL Image

# def overlay_segmentation_on_geotiff(image_path, results):
#     """
#     Apply segmentation overlays to a corresponding GeoTIFF image without changing geospatial information.
#     """
#     base_name = os.path.splitext(os.path.basename(image_path))[0]
#     geotiff_path = os.path.join(geotiff_dir, f"{base_name}.tif")
#     output_path = os.path.join(output_geotiff_dir, f"{base_name}_segmented.tif")
    
#     if not os.path.exists(geotiff_path):
#         print(f"Warning: Corresponding GeoTIFF not found for {image_path}")
#         return
    
#     with rasterio.open(geotiff_path) as src:
#         meta = src.meta.copy()
#         img_array = src.read()
#         mask_layer = np.zeros_like(img_array[0], dtype=np.uint8)

#         for result in results:
#             if result.masks is None:
#                 continue
#             for mask in result.masks.xy:
#                 cv2.fillPoly(mask_layer, [np.array(mask, dtype=np.int32)], 255)  # Fill with white

#         # Overlay mask on the first channel (assuming single-band TIFF or RGB first band)
#         img_array[0] = cv2.addWeighted(img_array[0], 1, mask_layer, 0.5, 0)

#         # Save modified GeoTIFF
#         with rasterio.open(output_path, "w", **meta) as dst:
#             dst.write(img_array)

# def display_images(original, predicted):
#     fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    
#     # Show Original Image
#     axes[0].imshow(original)
#     axes[0].set_title("Original Image")
#     axes[0].axis("off")
    
#     # Show Predicted Image
#     axes[1].imshow(predicted)
#     axes[1].set_title("Predicted Image")
#     axes[1].axis("off")
    
#     plt.show()

# # Process each image
# for image_path in image_paths:
#     # Load original image
#     img = Image.open(image_path).convert("RGB")
    
#     # Perform segmentation
#     results = model(image_path)

#     # Draw results with more control
#     pred_img = draw_segmentation_results(img, results)
    
#     # Overlay segmentation results on GeoTIFF
#     overlay_segmentation_on_geotiff(image_path, results)

#     # Display the original and predicted images
#     display_images(img, pred_img)


In [42]:
# geotiff_path = "m_3008501_ne_16_030_20211103.tif"  # Path to the input GeoTIFF file
# reassembled_output = "reassembled_output_tiff_yolo.tif"  # Path for the reassembled image
# output_dir = "output_geotiffs_yolo"           # Directory to save sliced images
# output_dir_normal = "output_slices"           # Directory to save sliced images

# print("Reassembling the image from slices...")
# reverse_slice_geotiff(input_dir=output_dir, output_geotiff=reassembled_output)

Reassembling the image from slices...


In [87]:
# Define unique colors for different classes
class_colors = {}

def get_class_color(cls):
    if cls not in class_colors:
        class_colors[cls] = tuple(np.random.randint(0, 255, 3).tolist())
    return class_colors[cls]

def draw_segmentation_results(image, results, alpha=0.5):
    """
    Draws segmentation masks, bounding boxes, and labels on the image with unique colors per class.
    """
    img_np = np.array(image)  # Convert PIL image to NumPy array
    for result in results:
        if result.masks is None or result.boxes is None:
            print("Warning: No segmentation masks found for this image.")
            return image  # Return the original image without changes

        for mask, box, cls, conf in zip(result.masks.xy, result.boxes.xyxy, result.boxes.cls, result.boxes.conf):
            cls = int(cls.item())
            color = get_class_color(cls)
            
            mask_np = np.zeros_like(img_np, dtype=np.uint8)
            cv2.fillPoly(mask_np, [np.array(mask, dtype=np.int32)], color)
            img_np = cv2.addWeighted(img_np, 1, mask_np, alpha, 0)

            x1, y1, x2, y2 = map(int, box.tolist())
            cv2.rectangle(img_np, (x1, y1), (x2, y2), color, 2)

            label = f"{model.names[cls]}: {conf:.2f}"  # Include confidence score
            text_x = x2 + 5
            text_y = y1 + 25  # Increase font size spacing
            cv2.putText(img_np, label, (text_x, text_y), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 3)

    return Image.fromarray(img_np)

# def overlay_segmentation_on_geotiff(image_path, results):
def overlay_segmentation_on_geotiff(results, geotiff_dir, output_geotiff_dir):
    """
    Apply segmentation overlays and bounding boxes to a corresponding GeoTIFF image without changing geospatial information.
    Returns a list of detected objects with their latitude, longitude, area, pixel count, class, and confidence.
    """
    base_name = os.path.splitext(os.path.basename(image_path))[0]
    geotiff_path = os.path.join(geotiff_dir, f"{base_name}.tif")
    output_path = os.path.join(output_geotiff_dir, f"{base_name}_segmented.tif")
    
    if not os.path.exists(geotiff_path):
        print(f"Warning: Corresponding GeoTIFF not found for {geotiff_path}")
        return []
    
    object_data = []
    
    with rasterio.open(geotiff_path) as src:
        meta = src.meta.copy()
        transform = src.transform
        img_array = src.read()
        mask_layer = np.zeros_like(img_array[0], dtype=np.uint8)

        for result in results:
            if result.masks is None:
                continue
            for mask, box, cls, conf in zip(result.masks.xy, result.boxes.xyxy, result.boxes.cls, result.boxes.conf):
                cls = int(cls.item())
                color = get_class_color(cls)
                
                cv2.fillPoly(mask_layer, [np.array(mask, dtype=np.int32)], 255)
                x1, y1, x2, y2 = map(int, box.tolist())
                cv2.rectangle(mask_layer, (x1, y1), (x2, y2), 255, 2)

                label = f"{model.names[cls]}: {conf:.2f}"
                text_x = x2 + 5
                text_y = y1 + 25
                cv2.putText(mask_layer, label, (text_x, text_y), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 3)
                
                # Calculate pixel count and area
                pixel_count = np.count_nonzero(mask_layer[y1:y2, x1:x2])
                area = pixel_count * (transform.a * abs(transform.e))  # Convert pixels to area using geotransform

                # Convert pixel coordinates to latitude and longitude
                lon, lat = transform * ((x1 + x2) / 2, (y1 + y2) / 2)
                object_data.append({
                    "latitude": lat, "longitude": lon, "area": area, "pixel_count": pixel_count,
                    "class": cls, "confidence": float(conf.item())
                })

        img_array[0] = cv2.addWeighted(img_array[0], 1, mask_layer, 0.5, 0)

        with rasterio.open(output_path, "w", **meta) as dst:
            dst.write(img_array)
    
    return object_data

def display_images(original, predicted):
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    
    axes[0].imshow(original)
    axes[0].set_title("Original Image")
    axes[0].axis("off")
    
    axes[1].imshow(predicted)
    axes[1].set_title("Predicted Image")
    axes[1].axis("off")
    
    plt.show()


In [68]:
print(accumulate_object_infos)

[{'latitude': 3430330.8, 'longitude': 602077.95, 'area': 952.74, 'pixel_count': 10586, 'class': 1, 'confidence': 0.6793378591537476}, {'latitude': 3430328.8499999996, 'longitude': 602109.75, 'area': 2062.08, 'pixel_count': 22912, 'class': 1, 'confidence': 0.8658507466316223}, {'latitude': 3430330.8, 'longitude': 602089.65, 'area': 555.39, 'pixel_count': 6171, 'class': 1, 'confidence': 0.8004984855651855}, {'latitude': 3430046.6999999997, 'longitude': 607218.0, 'area': 2048.04, 'pixel_count': 22756, 'class': 1, 'confidence': 0.9056617021560669}, {'latitude': 3430021.9499999997, 'longitude': 607214.4, 'area': 2061.09, 'pixel_count': 22901, 'class': 1, 'confidence': 0.8809728026390076}, {'latitude': 3429177.4499999997, 'longitude': 606994.65, 'area': 1401.84, 'pixel_count': 15576, 'class': 1, 'confidence': 0.8440890312194824}, {'latitude': 3429152.4, 'longitude': 606993.6, 'area': 1673.55, 'pixel_count': 18595, 'class': 1, 'confidence': 0.8246965408325195}, {'latitude': 3429518.25, 'longi

In [ ]:
import os
import cv2
import glob
import numpy as np
import torch
import matplotlib.pyplot as plt
from ultralytics import YOLO
import rasterio
from rasterio.transform import from_origin
from PIL import Image, ImageDraw, ImageFont

# Load the YOLO instance segmentation model
model = YOLO("C:/Users/mza0288/Documents/CODE/runs/segment/train29/weights/best.pt")

# Directories
image_dir = "C:/Users/mza0288/Documents/CODE/output_slices"  # RGB image directory
geotiff_dir = "C:/Users/mza0288/Documents/CODE/output_slices_geo"  # GeoTIFF directory
output_geotiff_dir = "C:/Users/mza0288/Documents/CODE/output_geotiffs_yolo"  # Output directory for modified GeoTIFFs

os.makedirs(output_geotiff_dir, exist_ok=True)

image_paths = glob.glob(os.path.join(image_dir, "*.jpg")) + glob.glob(os.path.join(image_dir, "*.png")) + glob.glob(os.path.join(image_dir, "*.jpeg"))


accumulate_object_infos = []

for image_path in image_paths:
    img = Image.open(image_path).convert("RGB")
    results = model(image_path)
    pred_img = draw_segmentation_results(img, results)
    object_info = overlay_segmentation_on_geotiff(image_path, results)
    display_images(img, pred_img)
    accumulate_object_infos.extend(object_info)
    print(f"Objects detected in {image_path}: {object_info}")

# Save accumulated object information to a CSV file
import pandas as pd
csv_output_path = os.path.join(output_geotiff_dir, "accumulated_object_infos.csv")
df = pd.DataFrame(accumulate_object_infos)
df.to_csv(csv_output_path, index=False)
print(f"Accumulated object information saved to {csv_output_path}")

In [ ]:
import os

# Define paths and parameters
geotiff_path = "m_3008501_ne_16_030_20211103.tif"  # Path to the input GeoTIFF file
output_dir = "output_slices_geo"           # Directory to save sliced images
output_dir_normal = "output_slices"           # Directory to save sliced images
reassembled_output = "reassembled_output_tiff_1.tif"  # Path for the reassembled image

output_size = 3000  # Size of each square slice in pixels
slice_flag = True  # Enable slicing

# Step 1: Slice the GeoTIFF image into tiles
print("Slicing the GeoTIFF image...")
slice_geotiff(
    geotiff_path=geotiff_path,
    output_dir=output_dir,
    output_size=output_size,
    slice_flag=slice_flag,  # Enable slicing
    to_jpeg=False            # Save slices as JPEG images
)

print("Slicing the normal image...")
slice_geotiff(
    geotiff_path=geotiff_path,
    output_dir=output_dir_normal,
    output_size=output_size,
    slice_flag=slice_flag,  # Enable slicing
    to_jpeg=True            # Save slices as JPEG images
)

In [88]:
import os
import cv2
import glob
import numpy as np
import torch
import matplotlib.pyplot as plt
from ultralytics import YOLO
import rasterio
from rasterio.transform import from_origin
from PIL import Image, ImageDraw, ImageFont

# Load the YOLO instance segmentation model
model = YOLO("C:/Users/mza0288/Documents/CODE/runs/segment/train29/weights/best.pt")

# Define paths and parameters
# reassembled_output = "reassembled_output_tiff_1.tif"  # Path for the reassembled image
image_dir = "C:/Users/mza0288/Documents/CODE/output_slices"  # RGB image directory
geotiff_dir = "C:/Users/mza0288/Documents/CODE/output_slices_geo"  # GeoTIFF directory
output_geotiff_dir = "C:/Users/mza0288/Documents/CODE/output_geotiffs_yolo"  # Output directory for modified GeoTIFFs
# geotiff_path = "m_3008501_ne_16_030_20211103.tif"  # Path to the input GeoTIFF file
output_dir = geotiff_dir        # Directory to save sliced images
output_dir_normal = image_dir          # Directory to save sliced images
os.makedirs(output_geotiff_dir, exist_ok=True)

output_size = 3000  # Size of each square slice in pixels
slice_flag = True  # Enable slicing

def find_tif_images(folder_path):
    return glob.glob(os.path.join(folder_path, '**', '*.tif'), recursive=True)


accumulate_object_infos = []

output_tile_size = (3000, 3000)
# folder_path = 'E:/Data/al_30cm_2021_transfer_1_of_2/31087/'
folder_path = 'C:/Users/mza0288/Documents/CODE/geotiff'
tif_images = find_tif_images(folder_path)
for tif_image in tif_images:
    print(f"Slicing the GeoTIFF image...{output_dir}")
    slice_geotiff(geotiff_path=tif_image,output_dir=output_dir,output_size=output_size,
        slice_flag=slice_flag,  # Enable slicing
        to_jpeg=False            # Save slices as JPEG images
    )
    print(f"Slicing the normal image...{output_dir_normal}")
    slice_geotiff(geotiff_path=tif_image,output_dir=output_dir_normal,output_size=output_size,
    slice_flag=slice_flag,  # Enable slicing
    to_jpeg=True            # Save slices as JPEG images
    )
    print(f"...Start Processing...{output_geotiff_dir}")
    image_paths = glob.glob(os.path.join(image_dir, "*.jpg")) + glob.glob(os.path.join(image_dir, "*.png")) + glob.glob(os.path.join(image_dir, "*.jpeg"))
    for image_path in image_paths:
        img = Image.open(image_path).convert("RGB")
        results = model(image_path)
        # pred_img = draw_segmentation_results(img, results)
        object_info = overlay_segmentation_on_geotiff(results, geotiff_dir, output_geotiff_dir)
        # display_images(img, pred_img)
        accumulate_object_infos.extend(object_info)
        # print(f"Objects detected in {image_path}: {object_info}")

# Save accumulated object information to a CSV file
import pandas as pd
csv_output_path = os.path.join("accumulated_object_infos_1.csv")
df = pd.DataFrame(accumulate_object_infos)
df.to_csv(csv_output_path, index=False)
print(f"Accumulated object information saved to {csv_output_path}")

Slicing the GeoTIFF image...{output_dir}
Slicing the normal image...{output_dir_normal}
...Start Processing...{output_geotiff_dir}

image 1/1 C:\Users\mza0288\Documents\CODE\output_slices\slice_0_0.jpg: 640x640 1 broiler house, 31.6ms
Speed: 4.4ms preprocess, 31.6ms inference, 6.3ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 C:\Users\mza0288\Documents\CODE\output_slices\slice_0_1.jpg: 640x640 2 broiler houses, 32.9ms
Speed: 3.8ms preprocess, 32.9ms inference, 5.5ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 C:\Users\mza0288\Documents\CODE\output_slices\slice_0_2.jpg: 640x640 (no detections), 33.9ms
Speed: 4.9ms preprocess, 33.9ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 C:\Users\mza0288\Documents\CODE\output_slices\slice_0_3.jpg: 640x640 (no detections), 32.8ms
Speed: 4.9ms preprocess, 32.8ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 C:\Users\mza0288\Documents\CODE\output_slices\slice_0